# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

In [ ]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

## Step 1: Target Variable Distribution

In [ ]:
fraud["fraud"].value_counts()

In [ ]:
fraud["fraud"].value_counts(normalize=True)

In [ ]:
fraud["fraud"].value_counts().plot(kind="bar")
plt.title("Fraud Distribution")
plt.xlabel("Fraud (0=legit, 1=fraud)")
plt.ylabel("Count")
plt.show()

Yes, we are dealing with an imbalanced dataset. Approximately 91% of transactions are legitimate (class 0) and only ~9% are fraudulent (class 1). This imbalance can cause a model to be biased toward predicting the majority class.

## Step 2 & 3: Baseline Logistic Regression (Imbalanced Data)

In [ ]:
X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

In [ ]:
pred = log_reg.predict(X_test_scaled)
print(classification_report(y_test, pred))

For fraud detection, **recall** is the most important metric for class 1 (fraud). Missing a fraudulent transaction (false negative) is far more costly than a false alarm. The baseline model tends to have low recall for the minority fraud class due to the class imbalance.

## Step 4: Oversampling

In [ ]:
train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
train_df["fraud"] = y_train.values

legit = train_df[train_df["fraud"] == 0]
fraud_minority = train_df[train_df["fraud"] == 1]

fraud_oversampled = resample(fraud_minority,
                             replace=True,
                             n_samples=len(legit),
                             random_state=42)

train_over = pd.concat([legit, fraud_oversampled])
print(train_over["fraud"].value_counts())

In [ ]:
train_over["fraud"].value_counts().plot(kind="bar")
plt.title("Fraud Distribution After Oversampling")
plt.show()

In [ ]:
X_train_over = train_over.drop(columns=["fraud"])
y_train_over = train_over["fraud"]

log_reg_over = LogisticRegression(max_iter=1000, random_state=42)
log_reg_over.fit(X_train_over, y_train_over)

pred_over = log_reg_over.predict(X_test_scaled)
print(classification_report(y_test, pred_over))

Oversampling the minority fraud class improves **recall for class 1 (fraud)**, meaning the model catches more fraudulent transactions. There may be a slight drop in precision since the model is more aggressive in flagging transactions as fraud.

## Step 5: Undersampling

In [ ]:
legit_undersampled = resample(legit,
                              replace=False,
                              n_samples=len(fraud_minority),
                              random_state=42)

train_under = pd.concat([legit_undersampled, fraud_minority])
print(train_under["fraud"].value_counts())

In [ ]:
train_under["fraud"].value_counts().plot(kind="bar")
plt.title("Fraud Distribution After Undersampling")
plt.show()

In [ ]:
X_train_under = train_under.drop(columns=["fraud"])
y_train_under = train_under["fraud"]

log_reg_under = LogisticRegression(max_iter=1000, random_state=42)
log_reg_under.fit(X_train_under, y_train_under)

pred_under = log_reg_under.predict(X_test_scaled)
print(classification_report(y_test, pred_under))

Undersampling also improves recall for class 1 (fraud), but the model is trained on a much smaller dataset, which may reduce its ability to generalise. The trade-off between recall and precision is similar to oversampling.

## Step 6: SMOTE

In [ ]:
sm = SMOTE(random_state=42, sampling_strategy=1.0)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

print(pd.Series(y_train_sm).value_counts())

In [ ]:
log_reg_sm = LogisticRegression(max_iter=1000, random_state=42)
log_reg_sm.fit(X_train_sm, y_train_sm)

pred_sm = log_reg_sm.predict(X_test_scaled)
print(classification_report(y_test, pred_sm))

SMOTE generates synthetic samples for the minority class using KNN, producing more diverse training data than simple oversampling. This typically yields a better balance between precision and recall for class 1 (fraud) compared to random oversampling, while avoiding the data loss of undersampling.

**Summary of results:**
- **Baseline (imbalanced):** High precision for fraud but low recall — many fraudulent transactions are missed.
- **Oversample:** Recall for fraud improves significantly; model catches more fraud.
- **Undersample:** Similar recall improvement to oversampling, but trained on less data.
- **SMOTE:** Best overall balance — improved recall for fraud with synthetic samples that generalise better than simple duplication.